## 🚀 PyTorch Segmentation Model Comparison 🚀

This notebook is a full conversion of the original TensorFlow/Keras notebook to PyTorch.

**Why the old one failed:** `tensorflow==2.15.0` (required by `segmentation-models`) does **not** support **Python 3.12**. 

**What this notebook does:**
1.  Uses `segmentation-models-pytorch` (smp), a much more stable library.
2.  Installs all dependencies cleanly with `pip`.
3.  Replaces the Keras `Sequence` with a PyTorch `Dataset` and `DataLoader`.
4.  Replicates the training, validation, and evaluation loop using PyTorch's manual-but-clearer style.
5.  Compares the same CNN-based models (U-Net, Attention U-Net, U-Net++).
6.  Uses Windows-compatible paths (`os.path.join`).

In [1]:
# --- Cell 1: Install PyTorch Dependencies ---

# 1. Install PyTorch with CUDA support. This command is for CUDA 12.1.
# If you have a different CUDA version, get the correct command from: https://pytorch.org/get-started/locally/
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Install the PyTorch segmentation library
!pip install segmentation-models-pytorch

# 3. Install other necessary libraries
!pip install pandas scikit-learn numpy pillow timm

Looking in indexes: https://download.pytorch.org/whl/cu121


In [2]:
# --- Cell 2: Imports and Global Configuration ---

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
from tqdm.notebook import tqdm
import gc

# --- Global Configuration ---
IMAGE_SIZE = (256, 256) 
BACKBONE = 'resnet34' 
BATCH_SIZE = 8 # Start with 8, increase to 16 if your GPU VRAM allows
N_CLASSES = 1
EPOCHS = 50
LEARNING_RATE = 1e-4

# Set device (GPU if available, else CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# --- Cell 3: Custom PyTorch Dataset ---

class SegDataset(Dataset):
    def __init__(self, frame_paths, mask_paths, image_size, preprocess_fn):
        self.frame_paths = frame_paths
        self.mask_paths = mask_paths
        self.image_size = image_size
        self.preprocess_fn = preprocess_fn
        
        # Basic transforms (Resize)
        self.resize_transform = T.Resize(image_size, interpolation=T.InterpolationMode.BILINEAR)
        self.mask_resize_transform = T.Resize(image_size, interpolation=T.InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.frame_paths)

    def __getitem__(self, idx):
        # Load image and mask
        frame_path = self.frame_paths[idx]
        mask_path = self.mask_paths[idx]
        
        image = Image.open(frame_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") # Grayscale

        # Resize
        image = self.resize_transform(image)
        mask = self.mask_resize_transform(mask)

        # Convert to numpy arrays
        image_np = np.array(image)
        mask_np = np.array(mask)

        # Apply backbone-specific preprocessing (normalization)
        if self.preprocess_fn:
            image_np = self.preprocess_fn(image_np)

        # Convert to tensors
        # PyTorch expects (Channels, Height, Width)
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1).float()
        
        # Scale mask to [0, 1] and add a channel dimension (Height, Width) -> (1, Height, Width)
        mask_tensor = torch.from_numpy(mask_np / 255.0).unsqueeze(0).float()
        
        return image_tensor, mask_tensor

In [4]:
# --- Cell 4: Data Preparation ---

# Define Paths (Using os.path.join for Windows compatibility)
FRAMES_DIR = os.path.join('archive', 'frames')
MASKS_DIR = os.path.join('archive', 'masks')

# Load and Sort Files
all_frames = sorted(glob.glob(os.path.join(FRAMES_DIR, '*.png')))
all_masks = sorted(glob.glob(os.path.join(MASKS_DIR, '*.png')))

if len(all_frames) == 0 or len(all_frames) != len(all_masks):
    print(f"FATAL ERROR: Data files not found or mismatched counts.")
    print(f"Checked for frames in: {os.path.abspath(FRAMES_DIR)}")
    print(f"Checked for masks in: {os.path.abspath(MASKS_DIR)}")
else:
    print(f"Total paired samples found: {len(all_frames)}")

# Split Data
train_frames, val_frames, train_masks, val_masks = train_test_split(
    all_frames, all_masks, test_size=0.2, random_state=42
)
print(f"Train samples: {len(train_frames)}, Validation samples: {len(val_frames)}")

# Get the specific preprocessing function for our ResNet backbone
preprocess_input = smp.encoders.get_preprocessing_fn(BACKBONE, pretrained='imagenet')

# Create Datasets
train_dataset = SegDataset(train_frames, train_masks, IMAGE_SIZE, preprocess_input)
val_dataset = SegDataset(val_frames, val_masks, IMAGE_SIZE, preprocess_input)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0) # num_workers=0 is safer on Windows
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Total paired samples found: 2729
Train samples: 2183, Validation samples: 546


In [5]:
# --- Cell 5: Loss Function and Metrics ---

import torch.nn as nn
import segmentation_models_pytorch as smp
# **FIX 1/2**: Import the functional API
from segmentation_models_pytorch.metrics import functional as F

# 1. Hybrid Loss (BCE + Dice)
# This class is correct and unchanged
class HybridLoss(nn.Module):
    def __init__(self, alpha=0.5, from_logits=True):
        super().__init__()
        self.alpha = alpha
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=from_logits)

    def forward(self, y_pred, y_true):
        bce = self.bce_loss(y_pred, y_true)
        dice = self.dice_loss(y_pred, y_true)
        return self.alpha * bce + (1 - self.alpha) * dice

criterion = HybridLoss().to(DEVICE)

# 2. Metrics
# **FIX 1/2**: Point to the functional metrics that accept tp, fp, fn, tn
METRICS = [
    F.iou_score,
    F.f1_score,       # This is Dice Coefficient
    F.sensitivity,
    F.specificity,
    F.accuracy
]

METRIC_NAMES = ['IoU', 'Dice Coefficient', 'Sensitivity', 'Specificity', 'Accuracy']

In [6]:
# --- Cell 6: PyTorch Training and Validation Loops ---

def train_epoch(model, loader, optimizer, criterion, device):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    # This loop is correct
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks) # criterion correctly handles logits
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

# **THIS FUNCTION IS NOW CORRECT**
def validate_epoch(model, loader, criterion, metrics, device):
    model.eval() # Set model to evaluation mode
    running_loss = 0.0
    
    # We will accumulate stats over the whole epoch (all are scalars)
    total_tp = torch.tensor(0.0, device=device)
    total_fp = torch.tensor(0.0, device=device)
    total_fn = torch.tensor(0.0, device=device)
    total_tn = torch.tensor(0.0, device=device)
    
    with torch.no_grad(): # Disable gradient calculations
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            # Forward pass (outputs are raw logits)
            outputs = model(images)
            
            # Calculate loss (criterion correctly handles logits)
            loss = criterion(outputs, masks)
            running_loss += loss.item() * images.size(0)
            
            # Apply sigmoid to logits to get probabilities for metrics
            probs = torch.sigmoid(outputs)
            
            # Calculate stats from probabilities
            # stats[0] will be a vector of shape [batch_size]
            stats = smp.metrics.get_stats(
                probs,          
                masks.long(),   
                mode='binary', 
                threshold=0.5
            )
            
            # **FIX**: Sum the stats from the batch before adding to the total
            total_tp += stats[0].sum()
            total_fp += stats[1].sum()
            total_fn += stats[2].sum()
            total_tn += stats[3].sum()

    epoch_loss = running_loss / len(loader.dataset)
    
    # Calculate final metrics from *accumulated scalar* stats
    epoch_metrics = []
    for metric_fn in metrics:
        metric_val = metric_fn(total_tp, total_fp, total_fn, total_tn, reduction='micro')
        epoch_metrics.append(metric_val.item())
    
    return epoch_loss, epoch_metrics

In [7]:
# --- Cell 7: Model Builder Functions ---

def build_unet():
    model = smp.Unet(
        encoder_name=BACKBONE,
        encoder_weights='imagenet',
        in_channels=3,
        classes=N_CLASSES,
    )
    model.name = 'U-Net_CNN'
    return model

def build_att_unet():
    model = smp.Unet(
        encoder_name=BACKBONE,
        encoder_weights='imagenet',
        in_channels=3,
        classes=N_CLASSES,
        attention_type='scse' # 'scse' is Squeeze-and-Excitation, a common attention mechanism
    )
    model.name = 'Attention_U-Net_CNN'
    return model

def build_unet_plus():
    model = smp.UnetPlusPlus(
        encoder_name=BACKBONE,
        encoder_weights='imagenet',
        in_channels=3,
        classes=N_CLASSES,
    )
    model.name = 'U-Net++_CNN'
    return model

# Note: TransUNet and Swin-UNet are not in 'smp' and require separate, complex libraries.
# We are focusing on the stable 'smp' models first.

In [8]:
# --- Cell 8: Main Training and Evaluation Orchestrator ---

def train_and_evaluate_model(model_builder_func, model_name):
    print(f"\n--- Starting Training for Model: {model_name} ---")
    
    # Clean up GPU memory
    torch.cuda.empty_cache()
    gc.collect()
    
    # 1. Build Model
    model = model_builder_func().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Use PyTorch's Learning Rate Scheduler (equivalent to ReduceLROnPlateau)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
    
    # 2. Training Loop
    best_val_dice = -1.0
    epochs_no_improve = 0
    patience = 10 # For Early Stopping
    best_model_path = os.path.join('.', f'{model_name}_best_model.pth')

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_metrics = validate_epoch(model, val_loader, criterion, METRICS, DEVICE)
        
        # Get Dice score for this epoch (it's the second metric in our list)
        current_val_dice = val_metrics[1]
        
        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Dice: {current_val_dice:.4f}")

        # Update LR Scheduler
        scheduler.step(val_loss)

        # Check for improvement (Save Best Model)
        if current_val_dice > best_val_dice:
            best_val_dice = current_val_dice
            torch.save(model.state_dict(), best_model_path)
            print(f"  New best model saved with Dice: {best_val_dice:.4f}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        # Check for Early Stopping
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    # 3. Final Evaluation
    print(f"Loading best weights for {model_name} from {best_model_path}")
    try:
        model.load_state_dict(torch.load(best_model_path))
    except Exception as e:
        print(f"Could not load saved weights, using last epoch. Error: {e}")

    # Evaluate the best model
    final_loss, final_metrics = validate_epoch(model, val_loader, criterion, METRICS, DEVICE)
    
    print(f"Evaluation Complete for {model_name}.")
    
    # Map results to a dictionary (same as original notebook)
    results_map = {
        'Model': model_name,
        'Loss (Hybrid)': final_loss
    }
    for name, value in zip(METRIC_NAMES, final_metrics):
        results_map[name] = value
        
    # Clean up model from memory
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return results_map

In [9]:
# --- Cell 9: EXECUTE ALL MODELS AND COMPARE ---

# List of model builders and their names
model_experiments = [
    (build_unet, "U-Net_CNN"),
    (build_att_unet, "Attention_U-Net_CNN"),
    (build_unet_plus, "U-Net++_CNN")
    # We can add transformer models here later if desired
]

all_results = []

for builder, name in model_experiments:
    try:
        results = train_and_evaluate_model(builder, name)
        all_results.append(results)
    except Exception as e:
        print(f"ERROR: Model {name} failed to train. Error: {e}")
        # This is often due to GPU OOM (Out of Memory).
        # If this happens, try reducing BATCH_SIZE in Cell 2.
        all_results.append({'Model': name, 'Loss (Hybrid)': np.nan, 'IoU': np.nan, 'Dice Coefficient': np.nan, 
                            'Sensitivity': np.nan, 'Specificity': np.nan, 'Accuracy': np.nan})
        torch.cuda.empty_cache()
        gc.collect()

# Convert all results to a DataFrame
df_results = pd.DataFrame(all_results)
if 'Model' in df_results.columns:
    df_results.set_index('Model', inplace=True)

# --- Final Output ---
print("\n" + "="*80)
print("                     🚀 FINAL PYTORCH MODEL COMPARISON (ResNet34 Backbone) 🚀")
print("="*80)

# Sort by the most critical metric (Dice Coefficient)
df_results_sorted = df_results.sort_values(by='Dice Coefficient', ascending=False)
print(df_results_sorted.to_markdown(floatfmt=".4f"))

# Optional: Save results
df_results_sorted.to_csv('segmentation_comparison_pytorch_results.csv')


--- Starting Training for Model: U-Net_CNN ---
Epoch 1/50 - Train Loss: 0.6044 - Val Loss: 0.4963 - Val Dice: 0.6063
  New best model saved with Dice: 0.6063
Epoch 2/50 - Train Loss: 0.4175 - Val Loss: 0.3390 - Val Dice: 0.7555
  New best model saved with Dice: 0.7555
Epoch 3/50 - Train Loss: 0.2583 - Val Loss: 0.2062 - Val Dice: 0.7910
  New best model saved with Dice: 0.7910
Epoch 4/50 - Train Loss: 0.1527 - Val Loss: 0.1445 - Val Dice: 0.8119
  New best model saved with Dice: 0.8119
Epoch 5/50 - Train Loss: 0.1204 - Val Loss: 0.1275 - Val Dice: 0.8108
Epoch 6/50 - Train Loss: 0.1014 - Val Loss: 0.1140 - Val Dice: 0.8252
  New best model saved with Dice: 0.8252
Epoch 7/50 - Train Loss: 0.0887 - Val Loss: 0.1060 - Val Dice: 0.8341
  New best model saved with Dice: 0.8341
Epoch 8/50 - Train Loss: 0.0827 - Val Loss: 0.1140 - Val Dice: 0.8180
Epoch 9/50 - Train Loss: 0.0810 - Val Loss: 0.1004 - Val Dice: 0.8401
  New best model saved with Dice: 0.8401
Epoch 10/50 - Train Loss: 0.0724 - 

C:\Users\Tahir\AppData\Local\Temp\ipykernel_20596\3419101024.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Evaluation Complete for U-Net_CNN.

--- Starting Training for Model: Attention_U-Net_CNN ---
Epoch 1/50 - Train Loss: 0.6500 - Val Loss: 0.5196 - Val Dice: 0.6298
  New best model saved with Dice: 0.6298
Epoch 2/50 - Train Loss: 0.4386 - Val Loss: 0.3508 - Val Dice: 0.7150
  New best model saved with Dice: 0.7150
Epoch 3/50 - Train Loss: 0.2643 - Val Loss: 0.2012 - Val Dice: 0.8016
  New best model saved with Dice: 0.8016
Epoch 4/50 - Train Loss: 0.1605 - Val Loss: 0.1454 - Val Dice: 0.8133
  New best model saved with Dice: 0.8133
Epoch 5/50 - Train Loss: 0.1199 - Val Loss: 0.1250 - Val Dice: 0.8222
  New best model saved with Dice: 0.8222
Epoch 6/50 - Train Loss: 0.0991 - Val Loss: 0.1158 - Val Dice: 0.8269
  New best model saved with Dice: 0.8269
Epoch 7/50 - Train Loss: 0.0897 - Val Loss: 0.1107 - Val Dice: 0.8299
  New best model saved with Dice: 0.8299
Epoch 8/50 - Train Loss: 0.0831 - Val Loss: 0.1043 - Val Dice: 0.8362
  New best model saved with Dice: 0.8362
Epoch 9/50 - Train 